<a href="https://colab.research.google.com/github/fhuang23/data-analyst-portfolio/blob/main/cafe_sales_clean_7_21_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training")

print("Path to dataset files:", path)

100%|██████████| 111k/111k [00:00<00:00, 540kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training/versions/1


In [ ]:
import os
path = "/root/.cache/kagglehub/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training/versions/1"
print(os.listdir(path))

['dirty_cafe_sales.csv']


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(f"{path}/dirty_cafe_sales.csv")  # filename may differ slightly

print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

df.head(50)

Loaded Dataset: 10000 rows, 8 columns


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [ ]:
# HEADER CLEANING #
print(df.columns.tolist())

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print("FIX APPLIED")
print(df.columns.tolist())

['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']
FIX APPLIED
['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent', 'payment_method', 'location', 'transaction_date']


In [ ]:
# TYPE CONVERSION & CURRENCY CLEANING #

print(df['item'].unique())
print(df['quantity'].unique())
print(df['price_per_unit'].unique())
print(df['total_spent'].unique())

['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan
 'ERROR' 'Juice' 'Tea']
['2' '4' '5' '3' '1' 'ERROR' 'UNKNOWN' nan]
['2.0' '3.0' '1.0' '5.0' '4.0' '1.5' nan 'ERROR' 'UNKNOWN']
['4.0' '12.0' 'ERROR' '10.0' '20.0' '9.0' '16.0' '15.0' '25.0' '8.0' '5.0'
 '3.0' '6.0' nan 'UNKNOWN' '2.0' '1.0' '7.5' '4.5' '1.5']


In [ ]:
print(df.loc[df['item'] == 'Cookie'].describe())

       transaction_id    item quantity price_per_unit total_spent  \
count            1092    1092     1075           1068        1073   
unique           1092       1        7              3           7   
top       TXN_7695629  Cookie        2            1.0         2.0   
freq                1    1092      231           1026         231   

        payment_method  location transaction_date  
count              829       762             1071  
unique               5         4              349  
top     Digital Wallet  Takeaway          UNKNOWN  
freq               265       362               21  


In [ ]:
columns = {
    'quantity',
    'price_per_unit',
    'total_spent'
}

In [ ]:
for col in columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[list(columns)].isnull().sum()

,0
price_per_unit,533
total_spent,502
quantity,479


In [ ]:
mask = df['total_spent'].isna() & df['quantity'].notna() & df['price_per_unit'].notna()
df.loc[mask, 'total_spent'] = df.loc[mask, 'quantity'] * df.loc[mask, 'price_per_unit']

In [ ]:
mask = df['price_per_unit'].isna() & df['quantity'].notna() & df['total_spent'].notna()
df.loc[mask, 'price_per_unit'] = df.loc[mask, 'total_spent'] / df.loc[mask, 'quantity']

In [ ]:
mask = df['quantity'].isna() & df['price_per_unit'].notna() & df['total_spent'].notna()
df.loc[mask, 'quantity'] = df.loc[mask, 'total_spent'] / df.loc[mask, 'price_per_unit']

In [ ]:
df[list(columns)].isnull().sum()

,0
price_per_unit,38
total_spent,40
quantity,38


In [ ]:
null_rows = df[df['item'].isna() | df['quantity'].isna() | df['price_per_unit'].isna() | df['total_spent'].isna()]
null_rows

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
8,TXN_4717867,NaN,5.0,3.0,15.0,NaN,Takeaway,2023-07-28
30,TXN_1736287,NaN,5.0,2.0,10.0,Digital Wallet,NaN,2023-06-02
61,TXN_8051289,NaN,1.0,3.0,3.0,NaN,In-store,2023-10-09
65,TXN_4987129,Sandwich,3.0,NaN,NaN,NaN,In-store,2023-10-20
72,TXN_6044979,NaN,1.0,1.0,1.0,Cash,In-store,2023-12-08
...,...,...,...,...,...,...,...,...
9869,TXN_1975184,Coffee,NaN,2.0,NaN,Digital Wallet,NaN,2023-01-15
9876,TXN_3105633,NaN,1.0,2.0,2.0,NaN,In-store,2023-03-30
9885,TXN_4659954,NaN,3.0,4.0,12.0,Credit Card,In-store,NaN
9893,TXN_3809533,Juice,2.0,NaN,NaN,Digital Wallet,Takeaway,2023-02-02


In [ ]:
item_dict = {'Cookie': 1.0, 'Tea': 1.5, 'Coffee': 2.0, 'Juice': 3.0, 'Cake': 3.0, 'Sandwich': 4.0, 'Smoothie': 4.0, 'Salad': 5.0}

In [ ]:
mask = df['price_per_unit'].isna()
df.loc[mask, 'price_per_unit'] = df.loc[mask, 'item'].map(item_dict)

In [ ]:
reverse_dict = {1.0: 'Cookie', 1.5: 'Tea', 2.0: 'Coffee', 3.0: 'Juice', 3.0: 'Cake', 4.0: 'Sandwich', 4.0: 'Smoothie', 5.0: 'Salad'}

In [ ]:
df['item'] = df['price_per_unit'].map(reverse_dict)

In [ ]:
df.isna().sum()

,0
transaction_id,0
item,0
quantity,0
price_per_unit,0
total_spent,0
payment_method,2564
location,3249
transaction_date,157


In [ ]:
df = df.dropna(subset=['item', 'quantity', 'price_per_unit', 'total_spent'])

In [ ]:
df[df['payment_method'].isin(['UNKNOWN', 'ERROR']) | df['payment_method'].isna()]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
6,TXN_4433211,Cake,3.0,3.0,9.0,ERROR,Takeaway,2023-10-06
8,TXN_4717867,Cake,5.0,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Smoothie,5.0,4.0,20.0,NaN,In-store,2023-12-31
13,TXN_9437049,Cookie,5.0,1.0,5.0,NaN,Takeaway,2023-06-01
...,...,...,...,...,...,...,...,...
9985,TXN_3297457,Cake,2.0,3.0,6.0,NaN,UNKNOWN,2023-01-03
9988,TXN_9594133,Cake,5.0,3.0,15.0,ERROR,NaN,NaN
9992,TXN_2739140,Smoothie,4.0,4.0,16.0,UNKNOWN,In-store,2023-07-05
9994,TXN_7851634,Smoothie,4.0,4.0,16.0,NaN,NaN,2023-01-08


In [ ]:
df[['payment_method', 'location', 'transaction_date']] = df[['payment_method', 'location', 'transaction_date']].replace(['ERROR', 'UNKNOWN'], np.nan)

In [ ]:
df = df.dropna(subset=['transaction_date'])

In [ ]:
#IMPUTE COLUMNS WITH NAN WITH RANDOM DISTRIBUTIONS#

def impute_random(df, column, seed=None):
    rng = np.random.default_rng(seed)
    known_values = df[column].dropna().values
    mask = df[column].isna()
    df.loc[mask, column] = rng.choice(known_values, size=mask.sum())
    return df

df = impute_random(df, 'payment_method', seed=42)
df = impute_random(df, 'location', seed=42)

In [ ]:
df.isna().sum()

,0
transaction_id,0
item,0
quantity,0
price_per_unit,0
total_spent,0
payment_method,0
location,0
transaction_date,0


In [ ]:
df.to_csv('final_result.csv', index=False)